In [1]:
import os
import json
from datetime import datetime
from openai import OpenAI
from dotenv import load_dotenv

# Initialize and pull sensitive API keys from your local environment setup securely
load_dotenv()

True

In [18]:
# =====================================================================
#  2.0 MANDATED BUSINESS TOOLS LIBRARY (FUNCTION CALLING)
# =====================================================================

def calculate(expression):
    """Calculate financial metrics, percentages, and growth rates safely."""
    try:
        # Safe execution environment stripping dangerous default Python built-ins
        result = eval(expression, {"__builtins__": {}}, {})
        return json.dumps({"result": result})
    except Exception as e:
        return json.dumps({"error": str(e)})

calculator_tool = {
    "type": "function",
    "function": {
        "name": "calculate",
        "description": "Calculate financial metrics, percentages, and growth rates using explicit arithmetic strings.",
        "parameters": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "Math expression string like '50000 * 0.15' or '(75000 - 50000) / 50000 * 100'"
                }
            },
            "required": ["expression"]
        }
    }
}


def web_search(query):
    """Search the web for business intelligence, industry news, market trends, or competitor info."""
    mock_results = {
        "market trends": "Tech sector showing 15% growth in Q1 2026 with rapid AI infrastructure expansion.",
        "industry news": "AI adoption increasing across all sectors, lifting administrative baselines by 22%.",
        "best practices": "Corporate communication standard: personalization, strict content scanning, and clear CTAs.",
        "competitor": "Main market competitors expanding global footprints to emerging secondary markets."
    }
    for keyword in mock_results:
        if keyword in query.lower():
            return json.dumps({"results": mock_results[keyword]})
    return json.dumps({"results": f"Verified current industry metrics regarding reference point: {query}"})

web_search_tool = {
    "type": "function",
    "function": {
        "name": "web_search",
        "description": "Search the web for business intelligence, industry news, market trends, or competitor info.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "The target business query string (e.g., 'market trends', 'competitor info')."
                }
            },
            "required": ["query"]
        }
    }
}


def analyze_data(data_string, operation):
    """Perform basic mathematical computations on raw data chunks or historical trends."""
    try:
        data = json.loads(data_string)
        if isinstance(data, list):
            values = [float(x) for x in data]
        elif isinstance(data, dict):
            values = [float(v) for v in data.values()]
        else:
            return json.dumps({"error": "Invalid format. Must be a JSON array string or dictionary wrapper object."})
            
        if operation == "sum":
            result = sum(values)
        elif operation == "average":
            result = sum(values) / len(values) if values else 0
        elif operation == "max":
            result = max(values) if values else 0
        elif operation == "min":
            result = min(values) if values else 0
        else:
            return json.dumps({"error": f"Unknown operational aggregation requirement: {operation}"})
            
        return json.dumps({"result": result, "count": len(values), "raw_processed": values})
    except Exception as e:
        return json.dumps({"error": str(e)})

data_analyzer_tool = {
    "type": "function",
    "function": {
        "name": "analyze_data",
        "description": "Perform basic mathematical computations on raw data arrays or historical business lists.",
        "parameters": {
            "type": "object",
            "properties": {
                "data_string": {
                    "type": "string",
                    "description": "A valid stringified JSON list of numbers or JSON object dict (e.g., '[50000, 65000]' or '{\"Jan\": 50000}')"
                },
                "operation": {
                    "type": "string",
                    "enum": ["sum", "average", "max", "min"],
                    "description": "The explicit computational aggregation process requested."
                }
            },
            "required": ["data_string", "operation"]
        }
    }
}

In [19]:
# =====================================================================
#   THE UNIFIED BUSINESS ASSISTANT ENGINE (ALL REQUIREMENTS MATCHED)
# =====================================================================

class BusinessEmailReportManager:
    def __init__(self):
        # Initialize native safe link with OpenAI endpoints
        self.client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
        
        # Tools configuration array mapping for Model usage pipelines
        self.tools = [calculator_tool, web_search_tool, data_analyzer_tool]
        self.functions = {
            "calculate": calculate,
            "web_search": web_search,
            "analyze_data": analyze_data
        }

    def _execute_tool_pipeline(self, messages, use_tools=True):
        """Internal recursive orchestration helper handling dynamic native agent execution."""
        if not use_tools:
            response = self.client.chat.completions.create(
                model="gpt-4o-mini",
                messages=messages
            )
            return response.choices[0].message.content

        response = self.client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=messages,
            tools=self.tools,
            tool_choice="auto"
        )
        
        response_message = response.choices[0].message
        
        # Intercept and process any explicit tool execution instructions from the model
        if response_message.tool_calls:
            messages.append(response_message)
            
            for tool_call in response_message.tool_calls:
                function_name = tool_call.function.name
                function_args = json.loads(tool_call.function.arguments)
                
                print(f" [Tool Call Executing]: calling '{function_name}' with {function_args}")
                tool_output = self.functions[function_name](**function_args)
                
                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": function_name,
                    "content": tool_output
                })
                
            # Recurse down path loop until the structural completion outputs are fully compiled
            return self._execute_tool_pipeline(messages, use_tools=True)
            
        return response_message.content

    # -----------------------------------------------------------------
    # 2.1 FEATURE 1: Smart Email Writer
    # -----------------------------------------------------------------
    def write_email(self, purpose, recipient, tone, research_topic=None):
        system_instruction = (
            "You are an expert email writer. Write personalized and structured business correspondence. "
            "Your output must STRICTLY follow this precise structure layout:\n"
            "1. Subject Line (Starting explicitly with 'Subject:')\n"
            "2. Professional Greeting (adapted exactly to the recipient type)\n"
            "3. Clean email body text containing between 3 to 5 comprehensive paragraphs\n"
            "4. A clean, professional closing block and designated signature wrapper line."
        )
        
        user_message = (
            f"Write a comprehensive business email based on these exact structural variables:\n"
            f"- Purpose/Goal: {purpose}\n"
            f"- Targeted Recipient Type: {recipient}\n"
            f"- Tone Required: {tone}\n"
        )
        if research_topic:
            user_message += f"- Optional Action: You MUST call the 'web_search' tool on the topic '{research_topic}' to gather industry insight before drafting."

        messages = [
            {"role": "system", "content": system_instruction},
            {"role": "user", "content": user_message}
        ]
        return self._execute_tool_pipeline(messages, use_tools=bool(research_topic))

    # -----------------------------------------------------------------
    # 2.2 FEATURE 2: Report Generator
    # -----------------------------------------------------------------
    def generate_report(self, report_type, data, period):
        system_instruction = (
            "You are a corporate reporting officer. You analyze data via execution tools and convert findings into elegant markdown layout records.\n"
            "You MUST use your 'analyze_data' tool to pull basic sums, averages, maximums, and minimums from raw dataset blocks.\n"
            "You MUST use your 'calculate' tool to evaluate percentages, growth trends, or performance scales if comparing shifts.\n\n"
            "Your finalized response must follow this EXACT markdown format outline:\n"
            "============================================\n"
            "[UPPERCASE REPORT TYPE] REPORT\n"
            "============================================\n"
            "Date: [Current Date or Specified Context Date]\n\n"
            "EXECUTIVE SUMMARY\n"
            "[2-3 sentences max summarizing highlights and high-level trends]\n\n"
            "KEY METRICS\n"
            "- Total Revenue: [Value]\n"
            "- Average Monthly: [Value]\n"
            "- Growth Rate: [Calculated Value]% QoQ\n"
            "- Best Month: [Name] ([Value])\n\n"
            "ANALYSIS\n"
            "[Detailed structural paragraph interpreting growth vectors, performance trends, and a descriptive chart visualization brief]\n\n"
            "RECOMMENDATIONS\n"
            "[List exactly 3 to 5 discrete bullet/numbered strategic business updates/points]"
        )
        
        user_message = (
            f"Generate a formal structural report for: {report_type}.\n"
            f"- Targeted Timeframe Chunk: {period}\n"
            f"- Dataset Profile Map: {json.dumps(data)}\n"
            f"Invoke your analytical tools now to parse out the required metrics before formatting."
        )
        
        messages = [
            {"role": "system", "content": system_instruction},
            {"role": "user", "content": user_message}
        ]
        return self._execute_tool_pipeline(messages, use_tools=True)

    # -----------------------------------------------------------------
    # 2.3 FEATURE 3: Meeting Summarizer
    # -----------------------------------------------------------------
    def summarize_meeting(self, notes, date, attendees="Not specified"):
        system_instruction = (
            "You are an administrative executive assistant. Parse messy, raw multi-line meeting notes into structured records.\n"
            "Your output framework MUST mirror this exact structural text asset layout:\n"
            "===========================================\n"
            "MEETING SUMMARY\n"
            "===========================================\n"
            "Date: [Specified Date]\n"
            "Attendees: [Attendees List]\n\n"
            "SUMMARY\n"
            "[2-3 sentence high level objective overview summarizing output resolutions]\n\n"
            "KEY POINTS\n"
            "• [Bullet points capturing core conversational pathways and discussion vectors]\n\n"
            "DECISIONS\n"
            "1. [Numbered list outlining finalized structural changes/approvals]\n\n"
            "ACTION ITEMS\n"
            "• [Owner Name]: [Explicit assignment details with timelines if mentioned]\n\n"
            "NEXT MEETING: [Extracted Date or 'Not scheduled']"
        )
        
        user_message = f"Process these raw meeting constraints:\n- Date: {date}\n- Attendees: {attendees}\n- Notes:\n{notes}"
        
        messages = [
            {"role": "system", "content": system_instruction},
            {"role": "user", "content": user_message}
        ]
        return self._execute_tool_pipeline(messages, use_tools=False)

    # -----------------------------------------------------------------
    # 2.4 FEATURE 4: Data Analyzer
    # -----------------------------------------------------------------
    def analyze_business_data(self, query, data):
        system_instruction = (
            "You are an expert business intelligence data analyst context-engine. Process natural language analytics requests over datasets.\n"
            "You MUST call your 'analyze_data' or 'calculate' tools to safely compute statistics.\n\n"
            "Your output formatting architecture must STRICTLY follow this schema shape:\n"
            "ANALYSIS RESULTS\n"
            "==================\n"
            "[Display direct answers to core parameters like Averages, Totals, Growth Scale %, and Directional Trend Vector]\n\n"
            "INTERPRETATION\n"
            "[A comprehensive paragraph detailing mathematical dynamics, velocity, stability, or comparative fluctuations]\n\n"
            "RECOMMENDATION\n"
            "[Actionable corporate advice to optimize performance based on historical numbers]"
        )
        
        user_message = f"Query: \"{query}\"\nTarget Raw Dataset: {json.dumps(data)}\nExecute computations and output the results structural map."
        
        messages = [
            {"role": "system", "content": system_instruction},
            {"role": "user", "content": user_message}
        ]
        return self._execute_tool_pipeline(messages, use_tools=True)

    # -----------------------------------------------------------------
    # 2.5 FEATURE 5: Client Communication Drafter
    # -----------------------------------------------------------------
    def draft_client_communication(self, comm_type, client, context, tone):
        system_instruction = (
            "You are a specialized CRM and client success communication champion. Draft structured external communications.\n"
            "Your response must perfectly organize around these core component sections:\n"
            "1. Professional Greeting tailored to the customer entity\n"
            "2. Context-appropriate body text explaining metrics, steps, proposals, or conditions cleanly\n"
            "3. A highly explicit, transparent, and actionable Call-To-Action (CTA)\n"
            "4. A professional corporate closing block and signature line."
        )
        
        user_message = (
            f"Draft a formalized customer asset matching these parameters:\n"
            f"- Communication Type Category: {comm_type}\n"
            f"- Client Targeted: {client}\n"
            f"- Background Context/Constraints: {context}\n"
            f"- Tone Architecture: {tone}"
        )
        
        messages = [
            {"role": "system", "content": system_instruction},
            {"role": "user", "content": user_message}
        ]
        return self._execute_tool_pipeline(messages, use_tools=False)

    # =====================================================================
    #  THE INTERFACE LAUNCHER AND NATURAL LANGUAGE ARGS PARSER
    # =====================================================================
    def process_request(self, request):
        """
        Dynamically extracts parameter variables from natural English prompts,
        determines the target requirement pathway, and automates asset generation.
        """
        routing_prompt = f"""
        Analyze the intent and text data parameters embedded in this user request.
        Determine which target assignment feature route matches best: 'email', 'report', 'summarize', 'analyze', or 'client'.
        
        Extract the values from the prompt into the JSON keys matched below. 
        If missing or required by the specification example schema profiles, populate them with realistic details matching the user context.
        
        Return a clean JSON payload mapping to this exact structure shape:
        {{
            "route": "email" | "report" | "summarize" | "analyze" | "client",
            "email_args": {{"purpose": "str", "recipient": "str", "tone": "str", "research_topic": "str or null"}},
            "report_args": {{"report_type": "str", "data": {{}}, "period": "str"}},
            "summarize_args": {{"notes": "str", "date": "str", "attendees": "str"}},
            "analyze_args": {{"query": "str", "data": []}},
            "client_args": {{"comm_type": "str", "client": "str", "context": "str", "tone": "str"}}
        }}
        
        User Request: "{request}"
        """
        
        response = self.client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "user", "content": routing_prompt}],
            response_format={"type": "json_object"}
        )
        
        extracted = json.loads(response.choices[0].message.content)
        route = extracted.get("route")
        
        print(f"\n🔀 [Unified Router Mapping]: Directing processing stream to -> FEATURE: '{route.upper()}'")
        
        if route == "email":
            return self.write_email(**extracted.get("email_args", {}))
        elif route == "report":
            return self.generate_report(**extracted.get("report_args", {}))
        elif route == "summarize":
            return self.summarize_meeting(**extracted.get("summarize_args", {}))
        elif route == "analyze":
            return self.analyze_business_data(**extracted.get("analyze_args", {}))
        elif route == "client":
            return self.draft_client_communication(**extracted.get("client_args", {}))
        else:
            return "Error: System failed to trace valid functional architecture pathways."


In [20]:
# =====================================================================
#  AUTOMATED DEPLOYMENT VERIFICATION SCENARIOS
# =====================================================================
if __name__ == "__main__":
    # Create master engine coordinator object inside safe verification runtimes
    manager = BusinessEmailReportManager()
    
    print("=================================================================")
    print("  RUNNING VERIFICATION FOR REQUIREMENT 2.1: SMART EMAIL WRITER")
    print("=================================================================")
    email_result = manager.write_email(
        purpose="Announce Q2 sales results to stakeholders",
        recipient="Stakeholders",
        tone="Formal",
        research_topic="market trends"
    )
    print(email_result)
    print("\n" + "#"*70 + "\n")

    print("=================================================================")
    print("   RUNNING VERIFICATION FOR REQUIREMENT 2.2: REPORT GENERATOR")
    print("=================================================================")
    report_result = manager.generate_report(
        report_type="Quarterly Sales Report",
        data={"Jan": 50000, "Feb": 65000, "Mar": 70000},
        period="Q1 2026"
    )
    print(report_result)
    print("\n" + "#"*70 + "\n")

    print("=================================================================")
    print("   RUNNING VERIFICATION FOR REQUIREMENT 2.3: MEETING SUMMARIZER")
    print("=================================================================")
    meeting_text = (
        "Discussed Q2 marketing campaign. Budget approved for $50k. "
        "Sarah will lead the campaign. Need to hire 2 designers by next month. "
        "Launch date set for June 1st. Ahmed to prepare creative brief. Next meeting May 10th."
    )
    meeting_result = manager.summarize_meeting(
        notes=meeting_text,
        date="May 3, 2026",
        attendees="Sarah, Ahmed, HR Team"
    )
    print(meeting_result)
    print("\n" + "#"*70 + "\n")

    print("=================================================================")
    print("     RUNNING VERIFICATION FOR REQUIREMENT 2.4: DATA ANALYZER")
    print("=================================================================")
    analysis_result = manager.analyze_business_data(
        query="What's the average monthly revenue and growth rate?",
        data=[50000, 55000, 60000, 65000, 70000, 75000]
    )
    print(analysis_result)
    print("\n" + "#"*70 + "\n")

    print("=================================================================")
    print(" RUNNING VERIFICATION FOR REQUIREMENT 2.5: CLIENT COMM DRAFTER")
    print("=================================================================")
    client_result = manager.draft_client_communication(
        comm_type="Project Proposal",
        client="ABC Technologies",
        context="Website redesign project, 3-month timeline, budget $50k",
        tone="Professional but friendly"
    )
    print(client_result)
    print("\n=================================================================")

  RUNNING VERIFICATION FOR REQUIREMENT 2.1: SMART EMAIL WRITER
 [Tool Call Executing]: calling 'web_search' with {'query': 'market trends'}
# Subject: Announcement of Q2 Sales Results to Stakeholders

Dear Stakeholders,

I hope this message finds you well. I am pleased to announce the results of our sales performance for the second quarter of 2026. Our team has worked diligently to achieve remarkable outcomes, and I am thrilled to share the details with you.

In Q2, we have witnessed substantial growth and progress across all key areas. Our sales figures have exceeded expectations, reflecting a positive trend in our revenue stream. The efforts put forth by our dedicated staff members have been instrumental in driving these commendable results. This quarter's performance underscores our commitment to excellence and continuous improvement.

Furthermore, I would like to highlight that our industry, particularly the tech sector, is experiencing significant growth, with a notable 15% increa